## Collectables Hack: OCS Mentor Dashboard (Solution)

The OCS **mentor dashboard** (`/mentor`) shows which capstone projects each mentor is on. To recruit the right mentors, OCS needs one answer from the mentor database: **which capstones still need a mentor, and in what area?** You will collect raw database rows into `AbstractCollectable` classes (from the [Abstract Collectables lesson](/java/collectable)) and let the `Registry` sort and group them.

Each hack below goes **problem, then solution, then result**. The problems match the [assignment](/java/collectable/mentor-dashboard).

## Hack #1: Expertise
### Problem
Build the `Expertise` attribute class. Extend `AbstractCollectable` with private `rank`, `area`, and `focus`. `getMap()` returns Rank, Area, Focus in that order; `compareTo()` orders by rank, not alphabetically; `catalog()` holds Frontend (0), Backend (1), Data & AI (2), and DevOps (3), added out of order. Show insertion order next to natural sort, and explain why rank order fits the dashboard.

### Solution

In [ ]:
// CODE_RUNNER: Hack #1 solution - Expertise
import java.util.*;

// ===== Lesson foundation (given, do not edit) =====

/** Contract for every dashboard record: project fields into a map, sort naturally. */
interface Collectable extends Comparable<Collectable> {
    Map<String, Object> getMap();
    List<String> getKeys();
    Object getValue(String key);
}

/** Shared code: subclasses only write getMap() and compareTo(). */
abstract class AbstractCollectable implements Collectable {
    public List<String> getKeys() { return new ArrayList<>(getMap().keySet()); }
    public Object getValue(String key) { return getMap().get(key); }
    public String toString() { return getMap().toString(); }
}

/** A typed list of collectables. */
class Catalog<T extends Collectable> {
    private final List<T> items = new ArrayList<>();
    public void add(T item) { items.add(item); }
    public List<T> getItems() { return items; }
    public void display() { items.forEach(System.out::println); }
}

/** Sorts, orders, and groups any catalog by its map keys. */
class Registry {
    private final List<Collectable> items;
    public Registry(Catalog<? extends Collectable> catalog) { items = new ArrayList<>(catalog.getItems()); }

    /** Prints the sort key first. */
    record ProjectedCollectable(Collectable delegate, String primaryKey) implements Collectable {
        public Map<String, Object> getMap() {
            Map<String, Object> original = delegate.getMap();
            Map<String, Object> reordered = new LinkedHashMap<>();
            if (original.containsKey(primaryKey)) reordered.put(primaryKey, original.get(primaryKey));
            original.forEach((k, v) -> { if (!k.equals(primaryKey)) reordered.put(k, v); });
            return reordered;
        }
        public List<String> getKeys() { return new ArrayList<>(getMap().keySet()); }
        public Object getValue(String key) { return getMap().get(key); }
        public int compareTo(Collectable o) { return delegate.compareTo(o); }
        public String toString() { return getMap().toString(); }
    }

    public List<Collectable> sortNatural() {
        List<Collectable> sorted = new ArrayList<>(items);
        Collections.sort(sorted);
        return sorted;
    }

    @SuppressWarnings({"unchecked", "rawtypes"})
    public List<Collectable> sortByField(String key) {
        List<Collectable> sorted = new ArrayList<>(items);
        sorted.sort((a, b) -> {
            Object valA = a.getValue(key), valB = b.getValue(key);
            return (valA instanceof Comparable && valB instanceof Comparable) ? ((Comparable) valA).compareTo(valB) : 0;
        });
        return sorted;
    }

    public List<ProjectedCollectable> orderBy(String key) {
        return sortByField(key).stream().map(item -> new ProjectedCollectable(item, key)).toList();
    }

    public Map<Object, List<Collectable>> groupBy(String key) {
        Map<Object, List<Collectable>> groups = new LinkedHashMap<>();
        for (Collectable item : items) groups.computeIfAbsent(item.getValue(key), k -> new ArrayList<>()).add(item);
        return groups;
    }
}

// ===== Hack #1: Expertise =====

/** A mentoring area. Natural order is rank, not alphabetical. */
class Expertise extends AbstractCollectable {
    private final int rank;
    private final String area;
    private final String focus;

    Expertise(int rank, String area, String focus) {
        this.rank = rank;
        this.area = area;
        this.focus = focus;
    }

    public String getArea() { return area; }

    public Map<String, Object> getMap() {
        Map<String, Object> map = new LinkedHashMap<>(); // keeps key order stable for printing
        map.put("Rank", rank);
        map.put("Area", area);
        map.put("Focus", focus);
        return map;
    }

    // Rank order follows the stack a capstone is built in: UI, API, data, deployment
    public int compareTo(Collectable o) { return Integer.compare(rank, (Integer) o.getValue("Rank")); }

    static Catalog<Expertise> catalog() {
        Catalog<Expertise> catalog = new Catalog<>(); // added out of order so the sort is visible
        catalog.add(new Expertise(2, "Data & AI", "analytics, ML, datasets"));
        catalog.add(new Expertise(0, "Frontend", "UI, SASS, accessibility"));
        catalog.add(new Expertise(3, "DevOps", "deployment, AWS, Docker"));
        catalog.add(new Expertise(1, "Backend", "APIs, Spring, databases"));
        return catalog;
    }

    /** Throws for an unknown area, so bad database rows get caught. (given) */
    static Expertise find(Catalog<Expertise> catalog, String area) {
        for (Expertise e : catalog.getItems()) if (e.getArea().equals(area)) return e;
        throw new IllegalArgumentException("unknown expertise " + area);
    }
}

// ===== Run Hack #1 =====
public class Main {
    public static void main(String[] args) {
        Catalog<Expertise> areas = Expertise.catalog();
        System.out.println("=== Expertise, insertion order ===");
        areas.display();
        System.out.println("\n=== Expertise, natural sort (rank) ===");
        new Registry(areas).sortNatural().forEach(System.out::println);

        System.out.println("\n=== Self-Check: Hack #1 ===");
        List<Object> sorted = new ArrayList<>();
        new Registry(areas).sortNatural().forEach(e -> sorted.add(e.getValue("Area")));
        check("4 areas in the catalog", areas.getItems().size() == 4);
        check("getMap() keys are Rank, Area, Focus",
              !areas.getItems().isEmpty() && areas.getItems().get(0).getKeys().equals(List.of("Rank", "Area", "Focus")));
        check("natural sort is Frontend, Backend, Data & AI, DevOps",
              sorted.equals(List.of("Frontend", "Backend", "Data & AI", "DevOps")));
        System.out.println(passed + "/" + total + " Hack #1 checks passed");
    }

    static int passed = 0, total = 0;

    static void check(String name, boolean ok) {
        total++;
        if (ok) passed++;
        System.out.println((ok ? "PASS " : "FAIL ") + name);
    }
}
Main.main(null);

### Result
Natural sort follows rank, not insertion order or the alphabet:

- Insertion order: Data & AI, Frontend, DevOps, Backend
- Natural sort:
  - `{Rank=0, Area=Frontend, Focus=UI, SASS, accessibility}`
  - `{Rank=1, Area=Backend, Focus=APIs, Spring, databases}`
  - `{Rank=2, Area=Data & AI, Focus=analytics, ML, datasets}`
  - `{Rank=3, Area=DevOps, Focus=deployment, AWS, Docker}`
- Self-Check: 3/3 Hack #1 checks passed

**Why it works:** `compareTo()` compares `Rank`, so `Collections.sort` inside `sortNatural()` puts the areas in the order a capstone is built: UI, API, data, deployment. Alphabetical order would put Backend first and DevOps before Frontend, which matches nothing on the dashboard. Adding them out of order proves the sort does the work, not the insertion order.

## Hack #2: Mentor
### Problem
Build the `Mentor` entity that has-a `Expertise`. `getOrg()` returns the email domain, `getOpenSlots()` returns capacity minus assigned, `getMap()` exposes Name, UID, Org, Expertise, Assigned, and Open Slots, and `compareTo()` sorts names ignoring case. Show Ada Rivera's row (`techcorp.example`, 1 open slot) and explain why derived fields belong in `getMap()`.

### Solution

In [ ]:
// CODE_RUNNER: Hack #2 solution - Mentor
import java.util.*;

// ===== Lesson foundation (given, do not edit) =====

/** Contract for every dashboard record: project fields into a map, sort naturally. */
interface Collectable extends Comparable<Collectable> {
    Map<String, Object> getMap();
    List<String> getKeys();
    Object getValue(String key);
}

/** Shared code: subclasses only write getMap() and compareTo(). */
abstract class AbstractCollectable implements Collectable {
    public List<String> getKeys() { return new ArrayList<>(getMap().keySet()); }
    public Object getValue(String key) { return getMap().get(key); }
    public String toString() { return getMap().toString(); }
}

/** A typed list of collectables. */
class Catalog<T extends Collectable> {
    private final List<T> items = new ArrayList<>();
    public void add(T item) { items.add(item); }
    public List<T> getItems() { return items; }
    public void display() { items.forEach(System.out::println); }
}

/** Sorts, orders, and groups any catalog by its map keys. */
class Registry {
    private final List<Collectable> items;
    public Registry(Catalog<? extends Collectable> catalog) { items = new ArrayList<>(catalog.getItems()); }

    /** Prints the sort key first. */
    record ProjectedCollectable(Collectable delegate, String primaryKey) implements Collectable {
        public Map<String, Object> getMap() {
            Map<String, Object> original = delegate.getMap();
            Map<String, Object> reordered = new LinkedHashMap<>();
            if (original.containsKey(primaryKey)) reordered.put(primaryKey, original.get(primaryKey));
            original.forEach((k, v) -> { if (!k.equals(primaryKey)) reordered.put(k, v); });
            return reordered;
        }
        public List<String> getKeys() { return new ArrayList<>(getMap().keySet()); }
        public Object getValue(String key) { return getMap().get(key); }
        public int compareTo(Collectable o) { return delegate.compareTo(o); }
        public String toString() { return getMap().toString(); }
    }

    public List<Collectable> sortNatural() {
        List<Collectable> sorted = new ArrayList<>(items);
        Collections.sort(sorted);
        return sorted;
    }

    @SuppressWarnings({"unchecked", "rawtypes"})
    public List<Collectable> sortByField(String key) {
        List<Collectable> sorted = new ArrayList<>(items);
        sorted.sort((a, b) -> {
            Object valA = a.getValue(key), valB = b.getValue(key);
            return (valA instanceof Comparable && valB instanceof Comparable) ? ((Comparable) valA).compareTo(valB) : 0;
        });
        return sorted;
    }

    public List<ProjectedCollectable> orderBy(String key) {
        return sortByField(key).stream().map(item -> new ProjectedCollectable(item, key)).toList();
    }

    public Map<Object, List<Collectable>> groupBy(String key) {
        Map<Object, List<Collectable>> groups = new LinkedHashMap<>();
        for (Collectable item : items) groups.computeIfAbsent(item.getValue(key), k -> new ArrayList<>()).add(item);
        return groups;
    }
}

// ===== Hack #1 solution =====

/** A mentoring area. Natural order is rank, not alphabetical. */
class Expertise extends AbstractCollectable {
    private final int rank;
    private final String area;
    private final String focus;

    Expertise(int rank, String area, String focus) {
        this.rank = rank;
        this.area = area;
        this.focus = focus;
    }

    public String getArea() { return area; }

    public Map<String, Object> getMap() {
        Map<String, Object> map = new LinkedHashMap<>(); // keeps key order stable for printing
        map.put("Rank", rank);
        map.put("Area", area);
        map.put("Focus", focus);
        return map;
    }

    // Rank order follows the stack a capstone is built in: UI, API, data, deployment
    public int compareTo(Collectable o) { return Integer.compare(rank, (Integer) o.getValue("Rank")); }

    static Catalog<Expertise> catalog() {
        Catalog<Expertise> catalog = new Catalog<>(); // added out of order so the sort is visible
        catalog.add(new Expertise(2, "Data & AI", "analytics, ML, datasets"));
        catalog.add(new Expertise(0, "Frontend", "UI, SASS, accessibility"));
        catalog.add(new Expertise(3, "DevOps", "deployment, AWS, Docker"));
        catalog.add(new Expertise(1, "Backend", "APIs, Spring, databases"));
        return catalog;
    }

    /** Throws for an unknown area, so bad database rows get caught. (given) */
    static Expertise find(Catalog<Expertise> catalog, String area) {
        for (Expertise e : catalog.getItems()) if (e.getArea().equals(area)) return e;
        throw new IllegalArgumentException("unknown expertise " + area);
    }
}

// ===== Hack #2: Mentor =====

/** A mentor record. HAS-A Expertise. */
class Mentor extends AbstractCollectable {
    private final String uid, name, email;
    private final Expertise expertise;
    private final int capacity, assigned;

    Mentor(String uid, String name, String email, Expertise expertise, int capacity, int assigned) {
        this.uid = uid;
        this.name = name;
        this.email = email;
        this.expertise = Objects.requireNonNull(expertise, "Mentor requires an Expertise");
        this.capacity = capacity;
        this.assigned = assigned;
    }

    public Expertise getExpertise() { return expertise; }

    // Derived fields: computed from stored data, so they can never go stale
    public String getOrg() { return email.substring(email.indexOf('@') + 1); }
    public int getOpenSlots() { return capacity - assigned; }

    public Map<String, Object> getMap() {
        Map<String, Object> map = new LinkedHashMap<>();
        map.put("Name", name);
        map.put("UID", uid);
        map.put("Org", getOrg());               // derived fields go in the map so the
        map.put("Expertise", expertise.getArea());
        map.put("Assigned", assigned);
        map.put("Open Slots", getOpenSlots());  // Registry can sort and group on them
        return map;
    }

    // Database names are not consistently capitalized ("frank kim"), so ignore case
    public int compareTo(Collectable o) { return name.compareToIgnoreCase((String) o.getValue("Name")); }
}

// ===== Run Hack #2 =====
public class Main {
    public static void main(String[] args) {
        Expertise backend = new Expertise(1, "Backend", "APIs, Spring, databases");
        Mentor ada = new Mentor("arivera", "Ada Rivera", "ada.rivera@techcorp.example", backend, 2, 1);
        Mentor frank = new Mentor("fkim", "frank kim", "frank@designlab.example", backend, 2, 0);
        Mentor grace = new Mentor("gnguyen", "Grace Nguyen", "grace@chainco.example", backend, 1, 0);

        System.out.println("=== Ada's dashboard row ===");
        System.out.println(ada);
        System.out.println("Org: " + ada.getValue("Org") + ", Open Slots: " + ada.getValue("Open Slots"));

        Catalog<Mentor> trio = new Catalog<>();
        trio.add(grace);
        trio.add(ada);
        trio.add(frank);
        System.out.println("\n=== Natural sort (name, ignoring case) ===");
        new Registry(trio).sortNatural().forEach(m -> System.out.println(m.getValue("Name")));

        System.out.println("\n=== Self-Check: Hack #2 ===");
        check("getMap() has Name, UID, Org, Expertise, Assigned, Open Slots",
              ada.getKeys().containsAll(List.of("Name", "UID", "Org", "Expertise", "Assigned", "Open Slots")));
        check("Org is the email domain", "techcorp.example".equals(ada.getValue("Org")));
        check("Open Slots = capacity - assigned", Integer.valueOf(1).equals(ada.getValue("Open Slots")));
        check("natural sort ignores case (frank kim before Grace)", frank.compareTo(grace) < 0);
        System.out.println(passed + "/" + total + " Hack #2 checks passed");
    }

    static int passed = 0, total = 0;

    static void check(String name, boolean ok) {
        total++;
        if (ok) passed++;
        System.out.println((ok ? "PASS " : "FAIL ") + name);
    }
}
Main.main(null);

### Result
- `{Name=Ada Rivera, UID=arivera, Org=techcorp.example, Expertise=Backend, Assigned=1, Open Slots=1}`
- Sorted: Ada Rivera, frank kim, Grace Nguyen
- Self-Check: 4/4 Hack #2 checks passed

**Why it works:** `Org` and `Open Slots` are computed from `email`, `capacity`, and `assigned`, so they can never disagree with the stored data. Putting them in `getMap()` is what lets the `Registry` sort and group on them later. **Edge case:** the database has "frank kim" in lowercase; plain `compareTo` would sort him after every capitalized name, so `compareToIgnoreCase` keeps the roster alphabetical.

## Hack #3: Mentor Dashboard
### Problem
Build the mentor dashboard with a `Registry`. `loadMentors` turns database rows into mentors and skips the bad Blockchain row without crashing. Print `orderBy("Open Slots")` and `groupBy("Expertise")`. `mentorGaps` groups unmentored capstones by Needs, subtracts open slots, and reports which areas OCS must recruit for (expected: 2 Backend). Explain that result in your own words.

### Solution

In [ ]:
// CODE_RUNNER: Hack #3 solution - Mentor Dashboard
import java.util.*;

// ===== Lesson foundation (given, do not edit) =====

/** Contract for every dashboard record: project fields into a map, sort naturally. */
interface Collectable extends Comparable<Collectable> {
    Map<String, Object> getMap();
    List<String> getKeys();
    Object getValue(String key);
}

/** Shared code: subclasses only write getMap() and compareTo(). */
abstract class AbstractCollectable implements Collectable {
    public List<String> getKeys() { return new ArrayList<>(getMap().keySet()); }
    public Object getValue(String key) { return getMap().get(key); }
    public String toString() { return getMap().toString(); }
}

/** A typed list of collectables. */
class Catalog<T extends Collectable> {
    private final List<T> items = new ArrayList<>();
    public void add(T item) { items.add(item); }
    public List<T> getItems() { return items; }
    public void display() { items.forEach(System.out::println); }
}

/** Sorts, orders, and groups any catalog by its map keys. */
class Registry {
    private final List<Collectable> items;
    public Registry(Catalog<? extends Collectable> catalog) { items = new ArrayList<>(catalog.getItems()); }

    /** Prints the sort key first. */
    record ProjectedCollectable(Collectable delegate, String primaryKey) implements Collectable {
        public Map<String, Object> getMap() {
            Map<String, Object> original = delegate.getMap();
            Map<String, Object> reordered = new LinkedHashMap<>();
            if (original.containsKey(primaryKey)) reordered.put(primaryKey, original.get(primaryKey));
            original.forEach((k, v) -> { if (!k.equals(primaryKey)) reordered.put(k, v); });
            return reordered;
        }
        public List<String> getKeys() { return new ArrayList<>(getMap().keySet()); }
        public Object getValue(String key) { return getMap().get(key); }
        public int compareTo(Collectable o) { return delegate.compareTo(o); }
        public String toString() { return getMap().toString(); }
    }

    public List<Collectable> sortNatural() {
        List<Collectable> sorted = new ArrayList<>(items);
        Collections.sort(sorted);
        return sorted;
    }

    @SuppressWarnings({"unchecked", "rawtypes"})
    public List<Collectable> sortByField(String key) {
        List<Collectable> sorted = new ArrayList<>(items);
        sorted.sort((a, b) -> {
            Object valA = a.getValue(key), valB = b.getValue(key);
            return (valA instanceof Comparable && valB instanceof Comparable) ? ((Comparable) valA).compareTo(valB) : 0;
        });
        return sorted;
    }

    public List<ProjectedCollectable> orderBy(String key) {
        return sortByField(key).stream().map(item -> new ProjectedCollectable(item, key)).toList();
    }

    public Map<Object, List<Collectable>> groupBy(String key) {
        Map<Object, List<Collectable>> groups = new LinkedHashMap<>();
        for (Collectable item : items) groups.computeIfAbsent(item.getValue(key), k -> new ArrayList<>()).add(item);
        return groups;
    }
}

// ===== OCS mentor database (given) =====

/** Rows as the database returns them. Row 7 is bad data on purpose. */
class MentorDatabase {
    // id, uid, name, email, expertise, capacity (max capstones), assigned (current capstones)
    static final Object[][] MENTOR_ROWS = {
        {1, "arivera", "Ada Rivera",    "ada.rivera@techcorp.example", "Backend",    2, 1},
        {2, "bchen",   "Ben Chen",      "ben@designlab.example",       "Frontend",   3, 3},
        {3, "cokafor", "Chioma Okafor", "chioma@datahub.example",      "Data & AI",  2, 0},
        {4, "dpatel",  "Dev Patel",     "dev@techcorp.example",        "Backend",    1, 1},
        {5, "elopez",  "Elena Lopez",   "elena@cloudops.example",      "DevOps",     2, 1},
        {6, "fkim",    "frank kim",     "frank@designlab.example",     "Frontend",   2, 0},
        {7, "gnguyen", "Grace Nguyen",  "grace@chainco.example",       "Blockchain", 1, 0},
    };
    // project, expertise it needs, mentors assigned
    static final Object[][] CAPSTONE_ROWS = {
        {"Food Bank Tracker",    "Backend",   0},
        {"Robotics Scheduler",   "DevOps",    1},
        {"Library Check-In",     "Frontend",  1},
        {"Volunteer Hours",      "Data & AI", 0},
        {"Tutor Match",          "Backend",   0},
        {"Club Fair Map",        "Frontend",  0},
        {"Senior Center Portal", "Backend",   0},
    };
}

/** A capstone project and the mentor expertise it needs. */
class Capstone extends AbstractCollectable {
    private final String project;
    private final String needs;
    private final int mentors;

    Capstone(String project, String needs, int mentors) {
        this.project = project;
        this.needs = needs;
        this.mentors = mentors;
    }

    public Map<String, Object> getMap() {
        Map<String, Object> map = new LinkedHashMap<>();
        map.put("Project", project);
        map.put("Needs", needs);
        map.put("Mentors", mentors);
        return map;
    }

    public int compareTo(Collectable o) { return project.compareTo((String) o.getValue("Project")); }

    static Catalog<Capstone> catalog() {
        Catalog<Capstone> catalog = new Catalog<>();
        for (Object[] row : MentorDatabase.CAPSTONE_ROWS) {
            catalog.add(new Capstone((String) row[0], (String) row[1], (Integer) row[2]));
        }
        return catalog;
    }
}

// ===== Hack #1 solution =====

/** A mentoring area. Natural order is rank, not alphabetical. */
class Expertise extends AbstractCollectable {
    private final int rank;
    private final String area;
    private final String focus;

    Expertise(int rank, String area, String focus) {
        this.rank = rank;
        this.area = area;
        this.focus = focus;
    }

    public String getArea() { return area; }

    public Map<String, Object> getMap() {
        Map<String, Object> map = new LinkedHashMap<>(); // keeps key order stable for printing
        map.put("Rank", rank);
        map.put("Area", area);
        map.put("Focus", focus);
        return map;
    }

    // Rank order follows the stack a capstone is built in: UI, API, data, deployment
    public int compareTo(Collectable o) { return Integer.compare(rank, (Integer) o.getValue("Rank")); }

    static Catalog<Expertise> catalog() {
        Catalog<Expertise> catalog = new Catalog<>(); // added out of order so the sort is visible
        catalog.add(new Expertise(2, "Data & AI", "analytics, ML, datasets"));
        catalog.add(new Expertise(0, "Frontend", "UI, SASS, accessibility"));
        catalog.add(new Expertise(3, "DevOps", "deployment, AWS, Docker"));
        catalog.add(new Expertise(1, "Backend", "APIs, Spring, databases"));
        return catalog;
    }

    /** Throws for an unknown area, so bad database rows get caught. (given) */
    static Expertise find(Catalog<Expertise> catalog, String area) {
        for (Expertise e : catalog.getItems()) if (e.getArea().equals(area)) return e;
        throw new IllegalArgumentException("unknown expertise " + area);
    }
}

// ===== Hack #2 solution =====

/** A mentor record. HAS-A Expertise. */
class Mentor extends AbstractCollectable {
    private final String uid, name, email;
    private final Expertise expertise;
    private final int capacity, assigned;

    Mentor(String uid, String name, String email, Expertise expertise, int capacity, int assigned) {
        this.uid = uid;
        this.name = name;
        this.email = email;
        this.expertise = Objects.requireNonNull(expertise, "Mentor requires an Expertise");
        this.capacity = capacity;
        this.assigned = assigned;
    }

    public Expertise getExpertise() { return expertise; }

    // Derived fields: computed from stored data, so they can never go stale
    public String getOrg() { return email.substring(email.indexOf('@') + 1); }
    public int getOpenSlots() { return capacity - assigned; }

    public Map<String, Object> getMap() {
        Map<String, Object> map = new LinkedHashMap<>();
        map.put("Name", name);
        map.put("UID", uid);
        map.put("Org", getOrg());               // derived fields go in the map so the
        map.put("Expertise", expertise.getArea());
        map.put("Assigned", assigned);
        map.put("Open Slots", getOpenSlots());  // Registry can sort and group on them
        return map;
    }

    // Database names are not consistently capitalized ("frank kim"), so ignore case
    public int compareTo(Collectable o) { return name.compareToIgnoreCase((String) o.getValue("Name")); }
}

// ===== Hack #3: Mentor Dashboard =====
public class Main {

    /** Turns database rows into Mentors. Skips rows with bad data instead of crashing. */
    static Catalog<Mentor> loadMentors(Object[][] rows, Catalog<Expertise> areas) {
        Catalog<Mentor> mentors = new Catalog<>();
        for (Object[] row : rows) {
            try {
                Expertise expertise = Expertise.find(areas, (String) row[4]);
                mentors.add(new Mentor((String) row[1], (String) row[2], (String) row[3],
                                       expertise, (Integer) row[5], (Integer) row[6]));
            } catch (IllegalArgumentException e) {
                // One bad row should not take down the whole dashboard
                System.out.println("Skipped row " + row[0] + ": " + e.getMessage());
            }
        }
        return mentors;
    }

    /** Area -> how many more mentors OCS needs (unmentored capstones minus open slots). */
    static Map<String, Integer> mentorGaps(Catalog<Capstone> capstones, Catalog<Mentor> mentors) {
        Catalog<Capstone> unmentored = new Catalog<>();
        for (Capstone c : capstones.getItems()) if ((Integer) c.getValue("Mentors") == 0) unmentored.add(c);

        Map<String, Integer> openSlots = new LinkedHashMap<>();
        for (Mentor m : mentors.getItems()) openSlots.merge(m.getExpertise().getArea(), m.getOpenSlots(), Integer::sum);

        Map<String, Integer> gaps = new LinkedHashMap<>();
        new Registry(unmentored).groupBy("Needs").forEach((area, projects) -> {
            int slots = openSlots.getOrDefault((String) area, 0);
            List<Object> names = new ArrayList<>();
            projects.forEach(p -> names.add(p.getValue("Project")));
            System.out.println(area + ": " + projects.size() + " unmentored " + names + ", " + slots + " open slot(s)");
            if (projects.size() > slots) gaps.put((String) area, projects.size() - slots);
        });
        return gaps;
    }

    public static void main(String[] args) {
        Catalog<Mentor> mentors = loadMentors(MentorDatabase.MENTOR_ROWS, Expertise.catalog());
        Registry registry = new Registry(mentors);

        System.out.println("\n=== 1. Mentors, natural sort (by Name) ===");
        registry.sortNatural().forEach(System.out::println);

        System.out.println("\n=== 2. Mentors, orderBy Open Slots ===");
        registry.orderBy("Open Slots").forEach(System.out::println);

        System.out.println("\n=== 3. Mentors, groupBy Expertise ===");
        registry.groupBy("Expertise").forEach((area, list) -> {
            List<Object> names = new ArrayList<>();
            list.forEach(m -> names.add(m.getValue("Name")));
            System.out.println(area + ": " + names);
        });

        System.out.println("\n=== 4. Where we need mentors ===");
        Map<String, Integer> gaps = mentorGaps(Capstone.catalog(), mentors);
        gaps.forEach((area, count) -> System.out.println("Recruit " + count + " " + area + " mentor(s)"));

        System.out.println("\n=== Self-Check: Hack #3 ===");
        check("6 mentors loaded, bad Blockchain row skipped (needs Hack #1 and #2 finished)", mentors.getItems().size() == 6);
        check("groupBy(\"Expertise\") gives 4 groups", registry.groupBy("Expertise").size() == 4);
        check("Backend needs 2 more mentors", Integer.valueOf(2).equals(gaps.get("Backend")));
        System.out.println(passed + "/" + total + " Hack #3 checks passed");
    }

    static int passed = 0, total = 0;

    static void check(String name, boolean ok) {
        total++;
        if (ok) passed++;
        System.out.println((ok ? "PASS " : "FAIL ") + name);
    }
}
Main.main(null);

### Result
- Skipped row 7: unknown expertise Blockchain
- `orderBy("Open Slots")`: Ben Chen 0, Dev Patel 0, Ada Rivera 1, Elena Lopez 1, Chioma Okafor 2, frank kim 2 (each row prints with `Open Slots` moved to the front)
- `groupBy("Expertise")`: Backend [Ada Rivera, Dev Patel], Frontend [Ben Chen, frank kim], Data & AI [Chioma Okafor], DevOps [Elena Lopez]
- `mentorGaps`:
  - Backend: 3 unmentored [Food Bank Tracker, Tutor Match, Senior Center Portal], 1 open slot(s)
  - Data & AI: 1 unmentored [Volunteer Hours], 2 open slot(s)
  - Frontend: 1 unmentored [Club Fair Map], 2 open slot(s)
- **Recruit 2 Backend mentor(s)**
- Self-Check: 3/3 Hack #3 checks passed

**Why it works:** `mentorGaps` reuses the `Registry`: it groups unmentored capstones by `Needs`, then subtracts each area's total open slots (`Map.merge` sums Ada's 1 and Dev's 0 for Backend). Backend has 3 projects but 1 slot, so OCS is short 2 Backend mentors; Frontend and Data & AI have more slots than projects, and DevOps has no unmentored project. **Edge case:** the Blockchain row throws in `Expertise.find()`, and `loadMentors` catches it, so one bad row is reported instead of crashing the dashboard.

## Reflection
- **Abstract class vs. interface:** `Collectable` is the contract and `AbstractCollectable` shares the code, so `Expertise`, `Mentor`, and `Capstone` each wrote only `getMap()` and `compareTo()`.
- **Has-a beats a String:** `Mentor` holds an `Expertise` object, so a typo like "Blockchain" fails in `find()` instead of quietly creating a fifth group.
- **The map drives the Registry:** `orderBy("Open Slots")` and `groupBy("Needs")` needed zero `Registry` changes because those values are in `getMap()`.
- **Next step for OCS:** connect `loadMentors` to the Spring `/api/capstone` mentor endpoints instead of hard-coded rows, so the dashboard updates when mentors are assigned.